# PS-P1 — Quantum-Safe Hybrid Key Exchange

This notebook is the reproducible control surface for the PS-P1 starter repository. Read `AGENTS.md`, `docs/PROTOCOL_SPEC.md`, and `docs/BENCHMARK_PROTOCOL.md` first. The complete benchmark is executed via scripts so raw CSV files and memory probes remain reproducible.


> **Security boundary:** this is a research/benchmark prototype, not a production secure channel. Baseline messages are unauthenticated and vulnerable to active man-in-the-middle attacks. Never run it with production secrets.

Platform B is a constrained container emulation, not physical embedded hardware. Run it only with verified CPU and memory limits.

In [ ]:
from pathlib import Path
import json, os, platform, subprocess, sys
ROOT = Path.cwd()
assert (ROOT / 'AGENTS.md').exists(), 'Run notebook from repository root'
print('Python:', sys.version)
print('OS:', platform.platform())
print('Repository root:', ROOT)


## 1. Hard preflight gate

This cell must succeed before ML-KEM-dependent cells run. A skipped ML-KEM test is not acceptance.

In [ ]:
preflight = subprocess.run([sys.executable, '-m', 'src.cli', 'check'], capture_output=True, text=True)
print(preflight.stdout)
if preflight.stderr:
    print(preflight.stderr, file=sys.stderr)
if preflight.returncode != 0:
    raise RuntimeError('Preflight gate failed. Install/repair native liboqs and verify ML-KEM-768; do not substitute legacy Kyber or mocks.')


## 2. Correctness smoke tests

The smoke test validates endpoint key agreement and expected raw payload sizes; it is not a performance sample.

In [ ]:
from src.handshake import run_handshake
expected_payload = {'ecdh': 130, 'mlkem': 2272, 'hybrid': 2402}
smoke = []
for suite, expected in expected_payload.items():
    result = run_handshake(suite)
    assert result['key_match'] is True
    assert result['total_payload_bytes'] == expected, (suite, result['total_payload_bytes'], expected)
    smoke.append({'suite': suite, 'key_match': result['key_match'], 'payload_bytes': result['total_payload_bytes']})
smoke


## 3. Tests

Run the formal test suite from a terminal. The command exits non-zero on failure. Keep stdout/stderr with the verification log.

In [ ]:
test_result = subprocess.run([sys.executable, '-m', 'pytest', '-q'], text=True, capture_output=True)
print(test_result.stdout)
if test_result.stderr:
    print(test_result.stderr, file=sys.stderr)
if test_result.returncode != 0:
    raise RuntimeError('Automated test gate failed')
print('Test command exit:', test_result.returncode)


## 4. Pilot benchmark (optional)

The pilot checks that CSV, manifest, and memory-probe outputs can be generated. Its sample size is too small for final claims. Set `RUN_PILOT = True` to run.

In [ ]:
RUN_PILOT = False
if RUN_PILOT:
    platform_id = os.environ.get('PS_P1_PLATFORM', 'A')
    pilot = subprocess.run([sys.executable, '-m', 'src.benchmark', '--platform', platform_id, '--iterations', '20', '--warmup', '5', '--batches', '1', '--memory-reps', '10'], text=True, capture_output=True)
    print(pilot.stdout)
    if pilot.stderr:
        print(pilot.stderr, file=sys.stderr)
    if pilot.returncode != 0:
        raise RuntimeError('Pilot benchmark failed; inspect diagnostics and do not proceed to final runs')


## 5. Final benchmark commands

Run the same source and container image for both platforms. The Platform B command must be run inside a container with enforced constraints. Do not label an unconstrained host run as Platform B.

```bash
# Platform A
python -m src.benchmark --platform A --iterations 1000 --warmup 100 --batches 5

# Platform B (run inside a constrained Linux container)
python -m src.benchmark --platform B --iterations 1000 --warmup 100 --batches 5

# Aggregate raw measurements and generate comparison plots
python scripts/analyze_results.py
```

Final Platform A/B commands are intentionally not executed by default from this notebook. Run the appropriate full benchmark only after tests pass and the platform is verified.